In [1]:
# importing basic libraries

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../src/data/raw/machine_failure.csv')

In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


# Data Cleaning & Preprocessing

In [4]:
df_cleaned = df.copy()

In [5]:
# Dropping the rows which our model not need.
df_cleaned.drop(columns=['UDI', 'Product ID', 'RNF'], inplace=True)

In [6]:
df_cleaned.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0


In [7]:
# Doing one hot encoding on Type

df_cleaned = pd.get_dummies(df_cleaned, columns=['Type'], prefix='Type', dtype=int, drop_first=True)

In [9]:
df_cleaned.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,1
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,1,0
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,1,0
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,1,0
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,1,0


In [11]:
df_cleaned.isna().sum() # all zero so need to drop any null and NaN column

Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
Type_L                     0
Type_M                     0
dtype: int64

In [15]:
df_cleaned['Air temperature [K]'].describe() # Its perfect no need to do anything on it its fine

count    10000.000000
mean       300.004930
std          2.000259
min        295.300000
25%        298.300000
50%        300.100000
75%        301.500000
max        304.500000
Name: Air temperature [K], dtype: float64

In [19]:
df_cleaned['Process temperature [K]'].describe() # Nothing to do cleaned data

count    10000.000000
mean       310.005560
std          1.483734
min        305.700000
25%        308.800000
50%        310.100000
75%        311.100000
max        313.800000
Name: Process temperature [K], dtype: float64

In [21]:
df_cleaned['Rotational speed [rpm]'].describe()

count    10000.000000
mean      1538.776100
std        179.284096
min       1168.000000
25%       1423.000000
50%       1503.000000
75%       1612.000000
max       2886.000000
Name: Rotational speed [rpm], dtype: float64

In [22]:
df_cleaned.shape

(10000, 12)

In [28]:
# Just reassign df_cleaned to the filtered version
df_cleaned = df_cleaned[
    (df_cleaned['Rotational speed [rpm]'] <= 2100) &
    (df_cleaned['Rotational speed [rpm]'] >= 1200)
]

print(f"Rows after filtering: {len(df_cleaned):,}") # 153 rows are removed 

Rows after filtering: 9,847


In [29]:
df_cleaned['Torque [Nm]'].describe()

count    9847.000000
mean       40.353763
std         9.539580
min        16.700000
25%        33.500000
50%        40.200000
75%        46.900000
max        76.600000
Name: Torque [Nm], dtype: float64

In [32]:
# I am only keeping 10-70 torque nothing else, all else are not legal and cant happen
df_cleaned = df_cleaned[
    (df_cleaned['Torque [Nm]'] >= 10) &
    (df_cleaned['Torque [Nm]'] <= 70 )
]

print(f"Rows after filtering: {len(df_cleaned):,}") # 12 rows are removed 

Rows after filtering: 9,835


In [33]:
df_cleaned.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,1
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,1,0
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,1,0
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,1,0
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,1,0


In [34]:
df_cleaned['Tool wear [min]'].describe()

count    9835.000000
mean      107.931368
std        63.609322
min         0.000000
25%        53.000000
50%       108.000000
75%       162.000000
max       253.000000
Name: Tool wear [min], dtype: float64

In [52]:
from sklearn.preprocessing import StandardScaler
from pandas.api.types import is_numeric_dtype
from typing import Union, List, Tuple, Dict, Any, Optional
import joblib
import os
import warnings

def standard_scaling(
    quan_col: Union[str, List[str]],
    dataframe: pd.DataFrame,
    save_scaler: bool = False,
    scaler_path: str = "scaler.pkl",
    return_copy: bool = False,
    inplace: bool = True
) -> Union[
    Tuple[str, StandardScaler, Dict[str, Any]],
    Tuple[pd.DataFrame, str, StandardScaler, Dict[str, Any]]
]:
    try:
        # VALIDATION
        if dataframe is None:
            raise ValueError("\nErr: dataframe can't be None!\n")

        if not isinstance(dataframe, pd.DataFrame):
            raise TypeError("\nErr: dataframe must be a pandas DataFrame.\n")

        if len(dataframe) == 0:
            raise ValueError("\nErr: length of dataframe can't be 0\n")

        # Convert to list if string
        if isinstance(quan_col, str):
            cols_to_scale = [quan_col]
        elif isinstance(quan_col, list):
            cols_to_scale = quan_col
        else:
            raise TypeError(
                f"\nErr: quan_col must be str or list, got {type(quan_col)}\n"
            )

        if not cols_to_scale:
            raise ValueError("\nErr: columns list cannot be empty\n")

        # Check all columns exist
        missing_cols = [col for col in cols_to_scale if col not in dataframe.columns]
        if missing_cols:
            raise KeyError(
                f"\nErr: columns not found: {missing_cols}\n"
                f"Available columns: {dataframe.columns.tolist()}"
            )

        # FIND AVAILABLE CONTINUOUS COLUMNS (for user feedback)
        continuous_cols: list[str] = [
            col for col in dataframe.columns
            if (
                is_numeric_dtype(dataframe[col])
                and dataframe[col].nunique() > 10
            )
        ]

        show_continuous_cols: str = (
            f"Available Continuous Columns: {continuous_cols}"
        )

        # VALIDATE EACH COLUMN
        for col in cols_to_scale:
            if not is_numeric_dtype(dataframe[col]):
                raise TypeError(
                    f"\nErr: column '{col}' is not numeric!\n"
                    f"{show_continuous_cols}"
                )

            if dataframe[col].nunique() <= 10:
                raise ValueError(
                    f"\nErr: column '{col}' is not continuous!\n"
                    f"(Only {dataframe[col].nunique()} unique values)\n"
                    f"{show_continuous_cols}"
                )

        # CREATE COPY IF RETURN_COPY IS TRUE
        if return_copy:
            df_to_scale = dataframe.copy()
            print("Working on a copy (original unchanged)")
        else:
            df_to_scale = dataframe
            print("Modifying dataframe in-place")

        # PERFORM STANDARD SCALING
        scaler: StandardScaler = StandardScaler()
        
        print(f"Scaling columns: {cols_to_scale}")
        
        # Scale ALL columns at once
        df_to_scale[cols_to_scale] = scaler.fit_transform(
            df_to_scale[cols_to_scale]
        )

        # BUILD METADATA
        metadata = {
            'columns': cols_to_scale,
            'n_columns': len(cols_to_scale),
            'n_samples': len(df_to_scale),
            'mean': scaler.mean_.tolist(),
            'scale': scaler.scale_.tolist(),
            'var': scaler.var_.tolist() if hasattr(scaler, 'var_') else None
        }

        # SAVE SCALER IF REQUESTED
        if save_scaler:
            os.makedirs(os.path.dirname(scaler_path) or '.', exist_ok=True)
            joblib.dump(scaler, scaler_path)
            save_msg = f"\nScaler saved to: {scaler_path}"
        else:
            save_msg = ""

        # BUILD SUMMARY
        summary = (
            f"\n{'='*50}\n"
            f"STANDARD SCALING COMPLETE\n"
            f"{'='*50}\n"
            f"Columns scaled: {cols_to_scale}\n"
            f"Number of columns: {len(cols_to_scale)}\n"
            f"Number of samples: {len(df_to_scale):,}\n"
            f"Mean: {[round(x, 4) for x in scaler.mean_.tolist()]}\n"
            f"Scale: {[round(x, 4) for x in scaler.scale_.tolist()]}\n"
            f"{save_msg}"
        )

        print(summary)

        # RETURN
        if return_copy:
            return (df_to_scale, summary, scaler, metadata)
        else:
            return (summary, scaler, metadata)

    except (ValueError, TypeError, KeyError) as e:
        print(f"\nError: {e}")
        raise

    except Exception as err:
        print(f"\nUnexpected error: {err}")
        raise RuntimeError(
            "An unexpected error occurred while performing standard scaling."
        ) from err

# We can do scaling task easily but i choose production function which makes you exceptional engineer and you will be able to solve 
# the real world tasks easily. 

In [59]:
import sys

sys.path.append(os.path.abspath('..'))

cols_for_scaling = [
    'Air temperature [K]', 
    'Process temperature [K]', 
    'Rotational speed [rpm]', 
    'Torque [Nm]', 
    'Tool wear [min]'
]

# Scale df_cleaned in-place
summary, scaler, metadata = standard_scaling(
    quan_col=cols_for_scaling,
    dataframe=df_cleaned,           # Scale df_cleaned (not df!)
    save_scaler=True,
    scaler_path='../src/data/processed/scaler.pkl'
)

# Now df_cleaned IS scaled!
print(df_cleaned[cols_for_scaling].head())

Modifying dataframe in-place
Scaling columns: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

STANDARD SCALING COMPLETE
Columns scaled: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Number of columns: 5
Number of samples: 9,835
Mean: [0.0, -0.0, 0.0, -0.0, -0.0]
Scale: [1.0, 1.0, 1.0, 1.0, 1.0]

Scaler saved to: ../src/data/processed/scaler.pkl
   Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0            -0.951735                -0.947020                0.160151   
1            -0.901673                -0.879534               -0.800851   
2            -0.951735                -1.014507               -0.196024   
3            -0.901673                -0.947020               -0.632844   
4            -0.901673                -0.879534               -0.800851   

   Torque [Nm]  Tool wear [min]  
0     0.262367        -1.696872  
1    

In [57]:
df_cleaned.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,Type_L,Type_M
0,-0.951735,-0.947020,0.160151,0.262367,-1.696872,0,0,0,0,0,0,1
1,-0.901673,-0.879534,-0.800851,0.631726,-1.649706,0,0,0,0,0,1,0
2,-0.951735,-1.014507,-0.196024,0.958873,-1.618263,0,0,0,0,0,1,0
3,-0.901673,-0.947020,-0.632844,-0.085885,-1.586819,0,0,0,0,0,1,0
4,-0.901673,-0.879534,-0.800851,-0.033120,-1.555376,0,0,0,0,0,1,0


In [61]:
# Creating final_df for model 
final_df = df_cleaned.copy()

In [62]:
# Define the path
file_path = '../src/data/processed/final_model_data.csv'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Save to CSV
final_df.to_csv(file_path, index=False)

print(f"CSV saved to: {file_path}")
print(f"Shape: {final_df.shape}")
print(f"File size: {os.path.getsize(file_path) / 1024:.2f} KB")

CSV saved to: ../src/data/processed/final_model_data.csv
Shape: (9835, 12)
File size: 1085.29 KB
